In [ ]:
!pip install --upgrade pip setuptools wheel
!pip install seqeval

  Using cached seqeval-1.2.2.tar.gz (43 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16251 sha256=8f67654b64a35eedd3b3405bee418cf3b7098260bd491d1352d71d148594c52b
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [ ]:
# ============================================================
# STAGE 1: DictaBERT-Large Training
# ============================================================
import os
import shutil
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seqeval_f1_score
from seqeval.metrics import precision_score as seqeval_precision
from seqeval.metrics import recall_score as seqeval_recall
from google.colab import drive
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# 1. Mount Google Drive
drive.mount('/content/drive')

BASE_MODEL_PATH = "/content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/"
LARGE_SAVE_PATH = "/content/drive/MyDrive/WORKSHOP/Large/models/stage1_dictabert_large/"
TMP_CHECKPOINT_DIR = "/tmp/large_train_checkpoints"

os.makedirs(LARGE_SAVE_PATH, exist_ok=True)

LABEL_2_ID = {"O": 0, "B-MU": 1, "I-MU": 2}
ID_2_LABEL = {0: "O", 1: "B-MU", 2: "I-MU"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 2. DATASET CLASS
# ============================================================
class GroupedStage1BIODataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len

        df = pd.read_csv(csv_path).dropna(subset=["Full_Raw_Sentence", "MeaningVal"])
        self.grouped_data = []

        for sent, group in df.groupby("Full_Raw_Sentence", sort=False):
            sentence = str(sent).strip()
            mv_spans = []
            search_start_pos = 0

            for _, row in group.iterrows():
                mv = str(row["MeaningVal"]).strip()
                if not mv:
                    continue
                start_pos = sentence.find(mv, search_start_pos)
                if start_pos == -1:
                    start_pos = sentence.find(mv)

                if start_pos != -1:
                    end_pos = start_pos + len(mv)
                    mv_spans.append({"text": mv, "start": start_pos, "end": end_pos})
                    search_start_pos = end_pos

            if mv_spans:
                self.grouped_data.append({"sentence": sentence, "meaning_spans": mv_spans})

    def __len__(self):
        return len(self.grouped_data)

    def __getitem__(self, idx):
        item = self.grouped_data[idx]
        sentence = item["sentence"]
        meaning_spans = item["meaning_spans"]

        encoding = self.tokenizer(
            sentence,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = encoding["offset_mapping"].squeeze(0).tolist()
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        valid_offsets = [o for o in offsets if o != [0, 0]]
        last_covered_char = valid_offsets[-1][1] if valid_offsets else 0
        is_truncated = last_covered_char < len(sentence)

        labels = [LABEL_2_ID["O"]] * len(offsets)
        for i, (start_tok, end_tok) in enumerate(offsets):
            if start_tok == 0 and end_tok == 0:
                labels[i] = -100

        sorted_spans = sorted(meaning_spans, key=lambda x: x["start"])
        for span in sorted_spans:
            mv_start, mv_end = span["start"], span["end"]
            if is_truncated and mv_start >= last_covered_char:
                continue

            first_token_matched = False
            for idx_tok, (start_tok, end_tok) in enumerate(offsets):
                if labels[idx_tok] == -100:
                    continue
                if not (end_tok <= mv_start or start_tok >= mv_end):
                    if not first_token_matched:
                        labels[idx_tok] = LABEL_2_ID["B-MU"]
                        first_token_matched = True
                    else:
                        labels[idx_tok] = LABEL_2_ID["I-MU"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# ============================================================
# 3. TRAINER & METRIC FUNCTIONS
# ============================================================
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = (
            torch.tensor(self.class_weights, dtype=torch.float).to(logits.device)
            if self.class_weights
            else None
        )
        loss_fct = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    preds = np.argmax(logits, axis=2)

    true_preds, true_labs = [], []
    for p_row, l_row in zip(preds, labels):
        c_preds, c_labs = [], []
        for p, l in zip(p_row, l_row):
            if l != -100:
                c_preds.append(ID_2_LABEL.get(int(p), "O"))
                c_labs.append(ID_2_LABEL.get(int(l), "O"))
        true_preds.append(c_preds)
        true_labs.append(c_labs)

    return {"span_f1": seqeval_f1_score(true_labs, true_preds)}

# ============================================================
# 4. TRAIN DICTABERT-LARGE
# ============================================================
MODEL_NAME = "dicta-il/dictabert-large"
print(f"\n📦 Loading Tokenizer and Model for {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID_2_LABEL, label2id=LABEL_2_ID, trust_remote_code=True
)

TRAIN_CSV = "train_set.csv" if os.path.exists("train_set.csv") else "stage1_splits/train_set.csv"
VAL_CSV = "val_set.csv" if os.path.exists("val_set.csv") else "stage1_splits/val_set.csv"

train_dataset = GroupedStage1BIODataset(TRAIN_CSV, tokenizer)
val_dataset = GroupedStage1BIODataset(VAL_CSV, tokenizer)

training_args = TrainingArguments(
    output_dir=TMP_CHECKPOINT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="span_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    class_weights=[1.0, 3.0, 2.0],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("\n🚀 Training DictaBERT-Large...")
trainer.train()

print(f"\n💾 Exporting final best model directly to Google Drive...")
model.save_pretrained(LARGE_SAVE_PATH)
tokenizer.save_pretrained(LARGE_SAVE_PATH)

shutil.rmtree(TMP_CHECKPOINT_DIR, ignore_errors=True)

print("=" * 65)
print(f"✅ MODEL SUCCESSFULLY SAVED TO DRIVE: {LARGE_SAVE_PATH}")
print("=" * 65)

# ============================================================
# 5. COMPARISON EVALUATION (BASE VS. LARGE)
# ============================================================
def evaluate_path(model_path, model_label, batch_size=8):
    print(f"\n⏳ Evaluating {model_label}...")

    try:
        eval_tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    except Exception:
        hub_id = "dicta-il/dictabert-large" if "Large" in model_label else "dicta-il/dictabert"
        eval_tok = AutoTokenizer.from_pretrained(hub_id, trust_remote_code=True)

    eval_mod = AutoModelForTokenClassification.from_pretrained(
        model_path, trust_remote_code=True
    ).to(device)
    eval_mod.eval()

    eval_ds = GroupedStage1BIODataset(VAL_CSV, eval_tok)
    dataloader = DataLoader(eval_ds, batch_size=batch_size)

    dummy = eval_tok(["מסלול שאפשר לנסוע"], return_tensors="pt").to(device)
    with torch.no_grad():
        _ = eval_mod(**dummy)

    all_preds, all_labs = [], []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].numpy()

            outputs = eval_mod(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

            for p_row, l_row in zip(preds, labels):
                seq_p, seq_l = [], []
                for p, l in zip(p_row, l_row):
                    if l != -100:
                        seq_p.append(ID_2_LABEL[int(p)])
                        seq_l.append(ID_2_LABEL[int(l)])
                all_preds.append(seq_p)
                all_labs.append(seq_l)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    f1 = seqeval_f1_score(all_labs, all_preds)
    precision = seqeval_precision(all_labs, all_preds)
    recall = seqeval_recall(all_labs, all_preds)
    latency_ms = ((end_time - start_time) / len(eval_ds)) * 1000

    del eval_mod, eval_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "Model": model_label,
        "Span F1 ↑": f"{f1*100:.2f}%",
        "Precision ↑": f"{precision*100:.2f}%",
        "Recall ↑": f"{recall*100:.2f}%",
        "Latency ↓": f"{latency_ms:.2f} ms"
    }

base_results = evaluate_path(BASE_MODEL_PATH, "DictaBERT Base")
large_results = evaluate_path(LARGE_SAVE_PATH, "DictaBERT Large")

df_slide = pd.DataFrame([base_results, large_results])

print("\n" + "=" * 65)
print("📊 FINAL SLIDE COMPARISON TABLE")
print("=" * 65)
print(df_slide.to_markdown(index=False))

Mounted at /content/drive

📦 Loading Tokenizer and Model for dicta-il/dictabert-large...


config.json:   0%|          | 0.00/407 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

✅ MODEL SUCCESSFULLY SAVED TO DRIVE: /content/drive/MyDrive/WORKSHOP/Large/models/stage1_dictabert_large/

⏳ Evaluating DictaBERT Base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


⏳ Evaluating DictaBERT Large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


📊 FINAL SLIDE COMPARISON TABLE
| Model           | Span F1 ↑   | Precision ↑   | Recall ↑   | Latency ↓   |
|:----------------|:------------|:--------------|:-----------|:------------|
| DictaBERT Base  | 75.27%      | 72.58%        | 78.17%     | 7.14 ms     |
| DictaBERT Large | 74.77%      | 71.17%        | 78.77%     | 23.54 ms    |


On Test

In [ ]:
# ============================================================
# STAGE 1: EVALUATION ONLY (BASE VS. LARGE)
# ============================================================
import os
import time
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from seqeval.metrics import f1_score as seqeval_f1_score
from seqeval.metrics import precision_score as seqeval_precision
from seqeval.metrics import recall_score as seqeval_recall
from google.colab import drive
from transformers import AutoModelForTokenClassification, AutoTokenizer

# 1. Mount Google Drive & Setup Paths
drive.mount('/content/drive', force_remount=False)

BASE_MODEL_PATH = "/content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/"
LARGE_SAVE_PATH = "/content/drive/MyDrive/WORKSHOP/Large/models/stage1_dictabert_large/"

TEST_CSV = "test_set.csv" if os.path.exists("test_set.csv") else "stage1_splits/test_set.csv"

LABEL_2_ID = {"O": 0, "B-MU": 1, "I-MU": 2}
ID_2_LABEL = {0: "O", 1: "B-MU", 2: "I-MU"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 2. DATASET CLASS
# ============================================================
class GroupedStage1BIODataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len

        df = pd.read_csv(csv_path).dropna(subset=["Full_Raw_Sentence", "MeaningVal"])
        self.grouped_data = []

        for sent, group in df.groupby("Full_Raw_Sentence", sort=False):
            sentence = str(sent).strip()
            mv_spans = []
            search_start_pos = 0

            for _, row in group.iterrows():
                mv = str(row["MeaningVal"]).strip()
                if not mv:
                    continue
                start_pos = sentence.find(mv, search_start_pos)
                if start_pos == -1:
                    start_pos = sentence.find(mv)

                if start_pos != -1:
                    end_pos = start_pos + len(mv)
                    mv_spans.append({"text": mv, "start": start_pos, "end": end_pos})
                    search_start_pos = end_pos

            if mv_spans:
                self.grouped_data.append({"sentence": sentence, "meaning_spans": mv_spans})

    def __len__(self):
        return len(self.grouped_data)

    def __getitem__(self, idx):
        item = self.grouped_data[idx]
        sentence = item["sentence"]
        meaning_spans = item["meaning_spans"]

        encoding = self.tokenizer(
            sentence,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = encoding["offset_mapping"].squeeze(0).tolist()
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        valid_offsets = [o for o in offsets if o != [0, 0]]
        last_covered_char = valid_offsets[-1][1] if valid_offsets else 0
        is_truncated = last_covered_char < len(sentence)

        labels = [LABEL_2_ID["O"]] * len(offsets)
        for i, (start_tok, end_tok) in enumerate(offsets):
            if start_tok == 0 and end_tok == 0:
                labels[i] = -100

        sorted_spans = sorted(meaning_spans, key=lambda x: x["start"])
        for span in sorted_spans:
            mv_start, mv_end = span["start"], span["end"]
            if is_truncated and mv_start >= last_covered_char:
                continue

            first_token_matched = False
            for idx_tok, (start_tok, end_tok) in enumerate(offsets):
                if labels[idx_tok] == -100:
                    continue
                if not (end_tok <= mv_start or start_tok >= mv_end):
                    if not first_token_matched:
                        labels[idx_tok] = LABEL_2_ID["B-MU"]
                        first_token_matched = True
                    else:
                        labels[idx_tok] = LABEL_2_ID["I-MU"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# ============================================================
# 3. EVALUATION FUNCTION
# ============================================================
def evaluate_path(model_path, model_label, data_path, batch_size=8):
    if not os.path.exists(model_path):
        print(f"⚠️ Model directory missing: {model_path}")
        return None

    print(f"\n⏳ Evaluating {model_label} on test set ({data_path})...")

    try:
        eval_tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    except Exception:
        hub_id = "dicta-il/dictabert-large" if "Large" in model_label else "dicta-il/dictabert"
        eval_tok = AutoTokenizer.from_pretrained(hub_id, trust_remote_code=True)

    eval_mod = AutoModelForTokenClassification.from_pretrained(
        model_path, trust_remote_code=True
    ).to(device)
    eval_mod.eval()

    eval_ds = GroupedStage1BIODataset(data_path, eval_tok)
    dataloader = DataLoader(eval_ds, batch_size=batch_size)

    dummy = eval_tok(["מסלול שאפשר לנסוע"], return_tensors="pt").to(device)
    with torch.no_grad():
        _ = eval_mod(**dummy)

    all_preds, all_labs = [], []

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].numpy()

            outputs = eval_mod(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

            for p_row, l_row in zip(preds, labels):
                seq_p, seq_l = [], []
                for p, l in zip(p_row, l_row):
                    if l != -100:
                        seq_p.append(ID_2_LABEL[int(p)])
                        seq_l.append(ID_2_LABEL[int(l)])
                all_preds.append(seq_p)
                all_labs.append(seq_l)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()

    f1 = seqeval_f1_score(all_labs, all_preds)
    precision = seqeval_precision(all_labs, all_preds)
    recall = seqeval_recall(all_labs, all_preds)
    latency_ms = ((end_time - start_time) / len(eval_ds)) * 1000

    del eval_mod, eval_tok
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "Model": model_label,
        "Span F1 ↑": f"{f1*100:.2f}%",
        "Precision ↑": f"{precision*100:.2f}%",
        "Recall ↑": f"{recall*100:.2f}%",
        "Latency ↓": f"{latency_ms:.2f} ms"
    }

# ============================================================
# 4. RUN COMPARISON ON TEST SET
# ============================================================
base_results = evaluate_path(BASE_MODEL_PATH, "DictaBERT Base", TEST_CSV)
large_results = evaluate_path(LARGE_SAVE_PATH, "DictaBERT Large", TEST_CSV)

eval_data = [res for res in [base_results, large_results] if res is not None]
df_slide = pd.DataFrame(eval_data)

print("\n" + "=" * 65)
print("📊 STAGE 1 TEST SET COMPARISON TABLE")
print("=" * 65)
print(df_slide.to_markdown(index=False))

Mounted at /content/drive

⏳ Evaluating DictaBERT Base on test set (test_set.csv)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


⏳ Evaluating DictaBERT Large on test set (test_set.csv)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


📊 STAGE 1 TEST SET COMPARISON TABLE
| Model           | Span F1 ↑   | Precision ↑   | Recall ↑   | Latency ↓   |
|:----------------|:------------|:--------------|:-----------|:------------|
| DictaBERT Base  | 75.90%      | 72.73%        | 79.36%     | 7.02 ms     |
| DictaBERT Large | 74.44%      | 70.94%        | 78.30%     | 25.64 ms    |
